# OpenMontage Kaggle Factory (Rewritten)This notebook installs selected models, generates TTS, images, and short b-roll videos for three channels: twistedtruths, crimeledger, mindtactics.Hardware target: Kaggle T4 (16GB) — use mixed precision and attention-slicing where available.

In [ ]:
# CELL 1: Install selected models and tools!pip install -q diffusers==0.18.0 transformers accelerate python-dotenv soundfile chatterbox-tts ffmpeg-python

In [ ]:
# CELL 2: Load IMAGE model (Stable Diffusion XL) on GPUimport torchfrom diffusers import StableDiffusionXLPipelinedevice = 'cuda' if torch.cuda.is_available() else 'cpu'print('Device:', device)sdxl = Nonetry:    sdxl = StableDiffusionXLPipeline.from_pretrained('stabilityai/stable-diffusion-xl-base-1.0', torch_dtype=torch.bfloat16)    sdxl.to(device)    print('SDXL loaded')except Exception as e:    print('SDXL load failed (falling back):', e)

In [ ]:
# CELL 3: Load VIDEO model (LTX-Video) on GPU — placeholderltx = Nonetry:    import ltx_video as ltx_module    ltx = getattr(ltx_module, 'load_model', lambda *a, **k: None)()    print('LTX-Video loader present (placeholder)')except Exception as e:    print('LTX-Video not found or failed to load (placeholder):', e)

In [ ]:
# CELL 4: Load Chatterbox TTS (assumes chatterbox-tts package)from dotenv import load_dotenvload_dotenv()try:    from chatterbox_tts import ChatterboxTTS    tts = ChatterboxTTS()    print('Chatterbox TTS loaded')except Exception as e:    tts = None    print('Chatterbox TTS failed to load:', e)

In [ ]:
# Helper utilitiesimport os, json, soundfile as sffrom pathlib import PathWORKDIR = Path.cwd()def ensure_dirs(ch):    base = WORKDIR / ch    for d in ['audio','images','video','render','shorts']:        (base / d).mkdir(parents=True, exist_ok=True)    return basedef save_wav(path, data, sr=24000):    sf.write(str(path), data, sr)

In [ ]:
# CHANNEL FUNCTION: runs TTS, IMAGE, VIDEO for a given projectimport subprocessdef process_channel(project_path, voice_style):    base = ensure_dirs(project_path)    script_path = Path('projects') / project_path / 'script.json'    if not script_path.exists():        print(f'script.json not found for {project_path}')        return    with script_path.open() as f:        script = json.load(f)    sections = script.get('sections', script.get('scenes', []))    # TTS per scene    for sec in sections:        sid = sec.get('id')        text = sec.get('text') or sec.get('narration') or ''        out_wav = base / 'audio' / f'{sid}.wav'        if tts is None:            open(out_wav, 'wb').close()        else:            wav = tts.generate(text, voice=voice_style)            save_wav(out_wav, wav, 24000)    # Images per scene (image_prompt or pexels_query)    PROMPT_SUFFIX = ', cinematic film still, dramatic chiaroscuro, dark teal amber palette, photorealistic, film noir, anamorphic bokeh, Arri Alexa, 8K, emotional tension'    for sec in sections:        sid = sec.get('id')        img_prompt = sec.get('image_prompt') or sec.get('speaker_directions') or sec.get('pexels_query','')        full_prompt = (img_prompt + ',' + PROMPT_SUFFIX).strip(',')        out_img = base / 'images' / f'{sid}.png'        if sdxl is None:            from PIL import Image            Image.new('RGB', (1344,768), (18,18,18)).save(out_img)        else:            im = sdxl(full_prompt, width=1344, height=768).images[0]            im.save(out_img)    # B-roll placeholder: make short clips from images    for sec in sections:        sid = sec.get('id')        img = base / 'images' / f'{sid}.png'        out_vid = base / 'video' / f'{sid}.mp4'        if img.exists():            cmd = f"ffmpeg -y -loop 1 -i {img} -c:v libx264 -t 3 -vf "scale=1920:1080,zoompan=z='if(lte(zoom,1.07),zoom+0.0008,zoom)'" -pix_fmt yuv420p {out_vid}"            try:                subprocess.run(cmd, shell=True, check=True)            except Exception as e:                print('ffmpeg failed for', sid, e)    print(f'Processed channel {project_path}')

In [ ]:
# CELL: Run channels in sequenceprocess_channel('twisted-truths-ep1', 'female_warm')import torch
torch.cuda.empty_cache()process_channel('crimeledger', 'serious_male')torch.cuda.empty_cache()process_channel('mindtactics', 'calm_authoritative_female')torch.cuda.empty_cache()

In [ ]:
# ZIP assets for each channelimport zipfilefor ch in ['twisted-truths-ep1','crimeledger','mindtactics']:    base = Path(ch)    if not base.exists(): continue    out = WORKDIR / f'{ch}_assets.zip'    with zipfile.ZipFile(out,'w') as zf:        for root,_,files in os.walk(base):            for f in files:                zf.write(os.path.join(root,f), arcname=os.path.relpath(os.path.join(root,f), WORKDIR))    print('Wrote', out)

In [ ]:
# SUMMARY: print countsfor ch in ['twisted-truths-ep1','crimeledger','mindtactics']:    base = Path(ch)    audio = len(list((base/'audio').glob('*.wav'))) if (base/'audio').exists() else 0    images = len(list((base/'images').glob('*.png'))) if (base/'images').exists() else 0    video = len(list((base/'video').glob('*.mp4'))) if (base/'video').exists() else 0    print(f'{ch}: {audio} audio, {images} images, {video} video')